# MassDOT IMPACT Crash Data Cleaning (2023–2025)

This notebook cleans the raw crash data so we can match crashes to road segments. The output is
the crash file we use for our label: did a road segment have at least one non-intersection crash
in 2023–2025?

Each step says what we remove and why. We save the row counts for every step to a cleaning log,
so we can show where every crash went.

**Input:** `data/raw/crashes_{year}.csv` 

**Output:**
- `data/processed/crashes_clean.csv`: the cleaned crashes
- `data/processed/cleaning_log.csv`: rows removed at each step
- `data/processed/excluded_by_junction.csv`: intersection/unknown crashes we set aside, by type

See `crash_data_exploration.ipynb` for the numbers behind these decisions.

In [ ]:
import time
import requests
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 50)

BASE = "https://gis.crashdata.dot.mass.gov/arcgis/rest/services/MassDOT"
SERVICES = {
    2023: "MASSDOT_ODP_OPEN_2023v",
    2024: "MASSDOT_ODP_OPEN_2024",
    2025: "MASSDOT_ODP_OPEN_2025",
}
LAYER = 0  # Crash layer, one row per crash

RAW_DIR = Path("data/raw")
OUT_DIR = Path("data/processed")
RAW_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Saving data under:", RAW_DIR.resolve().parent)

FIELDS = [
    "CRASH_NUMB", "CRASH_RPT_IDS", "CRASH_DATETIME", "YEAR", "CITY_TOWN_NAME",
    "X", "Y", "LAT", "LON", "RDWY", "NEAR_INT_RDWY", "DIST_DIRC_FROM_INT",
    "RDWY_JNCT_TYPE_DESCR", "CRASH_STATUS", "IS_GEOCODED", "GEOCODING_METHOD_NAME",
    "CRASH_SEVERITY_DESCR",
]

def layer_url(year):
    return f"{BASE}/{SERVICES[year]}/FeatureServer/{LAYER}"

def get_json(url, params=None, tries=3):
    for attempt in range(tries):
        try:
            r = requests.get(url, params=params, timeout=120)
            r.raise_for_status()
            data = r.json()
            if "error" in data:
                raise RuntimeError(data["error"])
            return data
        except Exception as e:
            if attempt == tries - 1:
                raise
            print(f"  retrying after error: {e}")
            time.sleep(3 * (attempt + 1))

## 1. Load the raw data

If the raw CSVs aren't there yet, this downloads them in pages of 2,000, the server's limit.
Each year is saved as soon as it finishes, so if the download gets cut off again we only lose
the year it was working on. Re-running skips years that are already saved.

In [ ]:
def download_year(year):
    url = f"{layer_url(year)}/query"
    rows, offset, page_size = [], 0, 2000
    while True:
        res = get_json(url, {
            "where": "1=1",
            "outFields": ",".join(FIELDS + ["OBJECTID"]),
            "returnGeometry": "false",
            "orderByFields": "OBJECTID",
            "resultOffset": offset,
            "resultRecordCount": page_size,
            "f": "json",
        })
        batch = [f["attributes"] for f in res.get("features", [])]
        if not batch:
            break
        rows.extend(batch)
        offset += len(batch)
        if offset % 20000 == 0:
            print(f"  {year}: {offset:,} records so far")
        if not res.get("exceededTransferLimit") and len(batch) < page_size:
            break
    df = pd.DataFrame(rows)
    df["CRASH_DATETIME"] = pd.to_datetime(df["CRASH_DATETIME"], unit="ms", errors="coerce")
    df["source_year"] = year
    return df

server_counts = {}
frames = []
for year in SERVICES:
    server_counts[year] = get_json(f"{layer_url(year)}/query",
        {"where": "1=1", "returnCountOnly": "true", "f": "json"})["count"]
    path = RAW_DIR / f"crashes_{year}.csv"
    if path.exists():
        df = pd.read_csv(path, parse_dates=["CRASH_DATETIME"])
        print(f"{year}: loaded {len(df):,} rows from {path}")
    else:
        print(f"{year}: downloading...")
        df = download_year(year)
        df.to_csv(path, index=False)
        print(f"{year}: saved {len(df):,} rows to {path}")
    frames.append(df)

raw = pd.concat(frames, ignore_index=True)
print(f"\nTotal raw crashes: {len(raw):,}")

### Checking the download

Our row counts should match what the server says it has. If a year doesn't match, some pages
didn't download, so delete that year's CSV and re-run.

In [ ]:
got = raw["source_year"].value_counts().sort_index()
pd.DataFrame({"downloaded": got, "server count": pd.Series(server_counts)})

## 2. Cleaning steps

We use one helper to log each step: how many rows we had before, how many after, and the reason.

In [ ]:
log = []

def step(df, keep, name, reason):
    """Keep the rows where `keep` is True and record what was removed."""
    out = df[keep].copy()
    log.append({"step": name, "reason": reason,
                "before": len(df), "removed": len(df) - len(out), "after": len(out)})
    print(f"{name}: removed {len(df) - len(out):,}, {len(out):,} left")
    return out

df = raw.copy()

### 2a. Remove duplicate crashes

The proposal says to deduplicate. `CRASH_NUMB` is the crash ID, so the same crash should never
show up twice. If it did, it would get counted twice when we check matching. We check within
each year and also across years, in case the same crash appears in two yearly services.

In [ ]:
dupes = df["CRASH_NUMB"].duplicated(keep="first")
print("Duplicate CRASH_NUMB rows:", int(dupes.sum()))
df = step(df, ~dupes, "Duplicates", "Same CRASH_NUMB appears more than once")

### 2b. Keep only geocoded crashes

We need a reliable location to match a crash to a road. We only keep `IS_GEOCODED = "Yes"`.
"No" means the crash was never placed on the map. "Multiple", "Low Confidence", and "One" are
uncertain placements, and they're only about 100 crashes per year, so we drop them too instead
of guessing. This step removes about 3–4% of crashes.

In [ ]:
print(df["IS_GEOCODED"].value_counts(dropna=False))
df = step(df, df["IS_GEOCODED"].eq("Yes"), "Geocoded",
          "No reliable location to match to a road")

### 2c. Check the coordinates

Even geocoded crashes could have missing or bad coordinates. We drop crashes that are missing X/Y,
or whose LAT/LON fall outside a box around Massachusetts (lat 41.2–42.9, lon -73.6 to -69.8).
A point outside the state can't match any road in the MassDOT inventory.

X/Y are in Massachusetts State Plane (EPSG 26986), which is in meters. That's what we want for
measuring crash-to-road distances, so we leave them as is.

In [ ]:
has_xy = df[["X", "Y"]].notna().all(axis=1)
in_ma = df["LAT"].between(41.2, 42.9) & df["LON"].between(-73.6, -69.8)
print("Missing X/Y:", int((~has_xy).sum()))
print("Outside Massachusetts box:", int((has_xy & ~in_ma).sum()))
df = step(df, has_xy & in_ma, "Coordinates",
          "Missing X/Y or location outside Massachusetts")

### 2d. Keep only non-intersection crashes

The label is about road segments, so we only keep crashes that aren't at a junction. A crash at an
intersection could belong to any of the roads that meet there, so there's no single segment to
assign it to. The proposal says to exclude intersection and unknown junction types and count
them separately, so we save those counts too.

**Driveway and ramp crashes:** these aren't intersections, but we haven't decided as a team
whether they count as segment crashes (about 6% of crashes). For now we only keep "Not at
junction". To include them, add them to `KEEP_JUNCTION` below and re-run.

In [ ]:
KEEP_JUNCTION = {"Not at junction"}
# Possible additions once the team decides: "Driveway", "On-ramp", "Off-ramp"

keep = df["RDWY_JNCT_TYPE_DESCR"].isin(KEEP_JUNCTION)

excluded_by_junction = (df.loc[~keep, "RDWY_JNCT_TYPE_DESCR"].fillna("(blank)")
                          .value_counts().rename("crashes").to_frame())
excluded_by_junction.to_csv(OUT_DIR / "excluded_by_junction.csv")
display(excluded_by_junction)

df = step(df, keep, "Non-intersection",
          "Intersection/unknown junction crashes can't be tied to one segment")

## 3. Flags (not removed)

These issues don't stop us from matching a crash, so we keep the rows but add a flag. That way
we can use them later in matching or in a sensitivity check.

- `missing_rdwy`: no reported road name. We can still match by distance, but we can't check that
  the road name agrees, so these matches are weaker.
- `missing_datetime`: no crash date/time. We only use the year, so this doesn't matter for the
  label, but it's good to know.
- `status_open`: the crash is still marked "Open". Every 2024 and 2025 crash is open, so those
  years may still change a little. 2025 has about 3% fewer crashes than the other years,
  probably because some reports haven't come in yet.

In [ ]:
df["missing_rdwy"] = df["RDWY"].isna() | df["RDWY"].astype(str).str.strip().eq("")
df["missing_datetime"] = df["CRASH_DATETIME"].isna()
df["status_open"] = df["CRASH_STATUS"].astype(str).str.startswith("Open")

df.groupby("source_year")[["missing_rdwy", "missing_datetime", "status_open"]].mean().mul(100).round(1)

## 4. Save and summarize

In [ ]:
clean_cols = ["CRASH_NUMB", "YEAR", "CRASH_DATETIME", "CITY_TOWN_NAME", "X", "Y", "LAT", "LON",
              "RDWY", "NEAR_INT_RDWY", "DIST_DIRC_FROM_INT", "RDWY_JNCT_TYPE_DESCR",
              "GEOCODING_METHOD_NAME", "CRASH_SEVERITY_DESCR", "CRASH_STATUS",
              "missing_rdwy", "missing_datetime", "status_open", "source_year"]
clean = df[clean_cols]
clean.to_csv(OUT_DIR / "crashes_clean.csv", index=False)

log_df = pd.DataFrame(log)
log_df.to_csv(OUT_DIR / "cleaning_log.csv", index=False)

print(f"Raw crashes:   {len(raw):,}")
print(f"Clean crashes: {len(clean):,} ({len(clean) / len(raw):.1%} kept)")
log_df

In [ ]:
by_year = pd.DataFrame({
    "raw": raw["source_year"].value_counts(),
    "clean": clean["source_year"].value_counts(),
}).sort_index()
by_year["% kept"] = (by_year["clean"] / by_year["raw"] * 100).round(1)
by_year

## 5. Summary

Most of the dropped crashes are intersection crashes, which we drop on purpose because they don't
belong to one segment. The data quality steps (duplicates, geocoding, coordinates) only remove a
few percent. `crashes_clean.csv` is ready to match to the 2022 road inventory.

Still to decide:
- Whether driveway and ramp crashes count (`KEEP_JUNCTION` in step 2d)
- Whether to keep 2025 as is, since it's still open and a bit low
- How to treat crashes with no road name during matching (`missing_rdwy`)